# Confidence Measure Metrics for VLM Evaluation

This notebook evaluates different confidence measures (e.g., Predictive Probability vs. Simulated Annotators) 
using Expected Calibration Error (ECE) and Area Under the ROC Curve (AUROC). 
This replicates the analysis from the original "Trust or Escalate" paper for the multimodal setup.


### How to Generate Results for Simulated Annotators
1. **Modify Config**: Open your YAML config (e.g., `configs/vlm_v2_guarantee.yaml`). 
   - Set `n_annotators: 1` in the `data` block for Predictive Probability.
   - Set `n_annotators: 3` (or higher) for Simulated Annotators.
2. **Setup Few-shot File**: Ensure the correct few-shot file is used in your dataset's `out_dir` (e.g., `data/vlm_v2_export`). 
   - For Simulated Annotators (Maj.), use `fewshot.maj.jsonl`. 
   - For Independent, use `fewshot.ind.jsonl`. 
   - Rename or symlink the desired file to `fewshot.jsonl`.
3. **Score**: Run the scoring pipeline: `python -m open_cascade score configs/<your_config>.yaml`.
4. **Organize Results**: Move the resulting `.jsonl` files from your default `result_dir` to a separate folder (e.g., `result/vlm_v2_maj/`) so you can compare them side-by-side below.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch

# Add root directory to python path
sys.path.append(str(Path("..").resolve()))

from open_cascade.cascade import merge_judgements
from open_cascade.metrics import ece, auroc


In [ ]:
# Configuration
JUDGE_MODELS = [
    "qwen2.5-vl-3b-instruct",
    "qwen2.5-vl-7b-instruct"
]

# We will map confidence measure names to the directory containing their results.
# Adjust the directory paths below to match where you have saved the results
# for each confidence measure technique.
RESULT_DIRS = {
    "Predictive Prob.": "../result/vlm_v2_prob",
    "Sim. Annot. (Ind.)": "../result/vlm_v2_ind", 
    "Sim. Annot. (Maj.)": "../result/vlm_v2_maj"
}


In [ ]:
def get_judgements_for_method(result_dir):
    """Load the cascade outputs for a specific confidence method."""
    import glob
    import json
    
    dir_path = Path(result_dir)
    if not dir_path.exists():
        print(f"Warning: Directory {result_dir} not found. Skipping.")
        return None
        
    samples = {}
    for model_name in JUDGE_MODELS:
        # We look for the evaluation jsonl file, e.g. test.jsonl or eval_pool.jsonl
        # The cascade script dumps output in <model_name>.*.jsonl
        jsonl_files = list(dir_path.glob(f"{model_name}.*.jsonl"))
        if not jsonl_files:
            print(f"Warning: No judgements found for {model_name} in {result_dir}")
            return None
            
        # Prioritize 'test' or 'eval_pool', else pick the first
        chosen_file = jsonl_files[0]
        for f in jsonl_files:
            if 'test' in f.name or 'eval_pool' in f.name:
                chosen_file = f
                break
                
        model_samples = []
        with open(chosen_file, 'r') as f:
            for line in f:
                model_samples.append(json.loads(line))
        samples[model_name] = model_samples
        
    phats, yhats, labels = merge_judgements(samples, JUDGE_MODELS)
    return phats, yhats, labels

results = []

for method_name, result_dir in RESULT_DIRS.items():
    res = get_judgements_for_method(result_dir)
    if res is None:
        continue
    phats, yhats, labels = res
    
    for model_name in JUDGE_MODELS:
        phat_tensor = phats[model_name]
        yhat_tensor = yhats[model_name]
        correct = (yhat_tensor == labels)
        
        # Calculate metrics
        model_ece = ece(phat_tensor, correct, n_bins=10)
        model_auroc = auroc(phat_tensor, correct)
        
        results.append({
            "Confidence Measure": method_name,
            "Judge Model": model_name,
            "ECE (↓)": round(model_ece, 3),
            "AUROC (↑)": round(model_auroc, 3)
        })


In [ ]:
# Display the comparison table
if results:
    df = pd.DataFrame(results)
    
    # Pivot to match the paper format: 
    # Rows: Confidence Measure
    # Columns: ECE and AUROC per Judge Model
    pivot_df = df.pivot(index="Confidence Measure", columns="Judge Model", values=["ECE (↓)", "AUROC (↑)"])
    
    # Reorder columns to group by Judge Model
    pivot_df = pivot_df.swaplevel(0, 1, axis=1).sort_index(axis=1, level=0)
    display(pivot_df)
else:
    print("No valid results found. Please check your RESULT_DIRS configuration.")
